# AC-MOT FINAL — Merge + Paper Tables

Run this AFTER the 3 parallel runs finish.

**Input:** 3 CSVs from Drive (Run1, Run2, Run3)  
**Output:**
- Main comparison table (A0 / A3 / BoT-SORT)
- Full ablation table (A0→A1→A2→A3→A4)
- Saved as 2 CSVs to Drive

In [ ]:
from pathlib import Path
from datetime import datetime
import pandas as pd, numpy as np
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DRIVE_RESULTS = Path('/content/drive/MyDrive/visdrone/VisDrone_Results')

# ── Find the 3 run CSVs automatically ──
run1 = sorted(DRIVE_RESULTS.glob('FINAL_RUN1_A0A1_*_per_seq.csv'),   key=lambda x: x.stat().st_mtime)[-1]
run2 = sorted(DRIVE_RESULTS.glob('FINAL_RUN2_A2A3_*_per_seq.csv'),   key=lambda x: x.stat().st_mtime)[-1]
run3 = sorted(DRIVE_RESULTS.glob('FINAL_RUN3_SOTA_*_per_seq.csv'),   key=lambda x: x.stat().st_mtime)[-1]

print(f'Run1: {run1.name}')
print(f'Run2: {run2.name}')
print(f'Run3: {run3.name}')

df_all = pd.concat([pd.read_csv(run1), pd.read_csv(run2), pd.read_csv(run3)], ignore_index=True)
print(f'\nTotal rows: {len(df_all)}')
print('Systems found:', sorted(df_all['system'].unique()))

In [ ]:
# ── Helper: summarize per system ──────────────────────────────────
def summarize(df, systems_order):
    rows = []
    for name in systems_order:
        g = df[df['system'] == name]
        if g.empty: continue
        rows.append(dict(
            system    = name,
            sequences = len(g),
            mota      = round(g['mota'].mean(), 4),
            idf1      = round(g['idf1'].mean(), 4),
            hota      = round(g['hota'].mean(), 4),
            recall    = round(g['recall'].mean(), 4),
            precision = round(g['precision'].mean(), 4),
            ids       = int(g['ids'].sum()),
            fn        = int(g['fn'].sum()),
            fp        = int(g['fp'].sum()),
            fps       = round(g['fps'].mean(), 1),
        ))
    out = pd.DataFrame(rows)
    if len(out) >= 2:
        base = out.iloc[0]
        out['mota_delta'] = (out['mota'] - float(base['mota'])).round(4)
        out['ids_delta']  = out['ids'] - int(base['ids'])
    return out

print('Summarize function ready')

In [ ]:
# ════════════════════════════════════════════════════════
# TABLE 1 — MAIN COMPARISON (for the paper Results section)
# ════════════════════════════════════════════════════════

# Include OC-SORT only if it ran
sota_systems = ['BoT-SORT']
if 'OC-SORT' in df_all['system'].values:
    sota_systems.append('OC-SORT')
    print('OC-SORT found in results ✅')
else:
    print('OC-SORT not in results (not supported) — using BoT-SORT only')

main_order = ['A0_Baseline_Default', 'A1_TunedTracker', 'A3_AdaptResolution'] + sota_systems
main_table = summarize(df_all, main_order)

print('\n' + '='*105)
print('TABLE 1 — MAIN COMPARISON | YOLOv8n backbone | 12 sequences | VisDrone2019-MOT-test-dev')
print('='*105)
print(main_table[['system','mota','idf1','hota','recall','precision','ids','fps','mota_delta','ids_delta']]
      .to_string(index=False))
print('='*105)

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
p1 = DRIVE_RESULTS / f'FINAL_table1_main_{ts}.csv'
main_table.to_csv(p1, index=False)
print(f'Saved -> {p1.name}')

In [ ]:
# ════════════════════════════════════════════════════════
# TABLE 2 — ABLATION STUDY (for the paper Ablation section)
# ════════════════════════════════════════════════════════
abl_order = ['A0_Baseline_Default','A1_TunedTracker','A2_AdaptThreshold','A3_AdaptResolution','A4_ReID']
abl_table = summarize(df_all, abl_order)

print('\n' + '='*100)
print('TABLE 2 — ABLATION STUDY | YOLOv8n fixed | 12 sequences')
print('='*100)
print(abl_table[['system','mota','idf1','hota','ids','fps','mota_delta','ids_delta']]
      .to_string(index=False))
print('='*100)

print('\nIncremental gain per step:')
steps = [
    ('A0→A1', 'Tuned ByteTrack YAML'),
    ('A1→A2', 'Adaptive Confidence Threshold'),
    ('A2→A3', 'Adaptive Resolution  ← expected dominant gain'),
    ('A3→A4', 'ReID  ← expected negative'),
]
for i, (label, desc) in enumerate(steps):
    if i+1 >= len(abl_table): break
    p = abl_table.iloc[i]; c = abl_table.iloc[i+1]
    dm   = float(c['mota']) - float(p['mota'])
    di   = float(c['idf1']) - float(p['idf1'])
    dids = int(c['ids'])    - int(p['ids'])
    flag = '✓' if (dm > 0 and dids <= 0) else ('~' if dm > 0 else '✗')
    print(f'  {label}  {desc:<46}  ΔMOTA={dm:+.4f}  ΔIDF1={di:+.4f}  ΔIDS={dids:+4d}  {flag}')

# Save
p2 = DRIVE_RESULTS / f'FINAL_table2_ablation_{ts}.csv'
abl_table.to_csv(p2, index=False)
print(f'\nSaved -> {p2.name}')

In [ ]:
# ════════════════════════════════════════════════════════
# PAPER COPY-PASTE SUMMARY
# ════════════════════════════════════════════════════════
base = main_table[main_table['system']=='A0_Baseline_Default'].iloc[0]
ours = main_table[main_table['system']=='A3_AdaptResolution'].iloc[0]
sota = main_table[main_table['system']=='BoT-SORT'].iloc[0]

print('\n' + '='*70)
print('PAPER NUMBERS (copy-paste ready)')
print('='*70)
print(f"AC-MOT vs Baseline:")
print(f"  MOTA  {float(base['mota']):.4f} → {float(ours['mota']):.4f}  ({float(ours['mota'])-float(base['mota']):+.4f} / {(float(ours['mota'])-float(base['mota']))/float(base['mota'])*100:+.1f}%)")
print(f"  IDF1  {float(base['idf1']):.4f} → {float(ours['idf1']):.4f}  ({float(ours['idf1'])-float(base['idf1']):+.4f})")
print(f"  HOTA  {float(base['hota']):.4f} → {float(ours['hota']):.4f}  ({float(ours['hota'])-float(base['hota']):+.4f})")
print(f"  IDS   {int(base['ids'])} → {int(ours['ids'])}  ({int(ours['ids'])-int(base['ids']):+d} / {(int(ours['ids'])-int(base['ids']))/int(base['ids'])*100:+.1f}%)")
print(f"  FPS   {float(base['fps']):.1f} → {float(ours['fps']):.1f}")
print(f"\nAC-MOT vs BoT-SORT (SOTA):")
print(f"  MOTA  AC-MOT={float(ours['mota']):.4f}  BoT-SORT={float(sota['mota']):.4f}")
print(f"  IDF1  AC-MOT={float(ours['idf1']):.4f}  BoT-SORT={float(sota['idf1']):.4f}")
print(f"  FPS   AC-MOT={float(ours['fps']):.1f}  BoT-SORT={float(sota['fps']):.1f}  ({float(ours['fps'])/float(sota['fps']):.1f}x faster)")
print('='*70)